# Lezione 10 · Il post-training: dal sapere al comportamento

Il giudice che impara dalle bandierine, la cima, la soglia dell'esame, la media dei fratelli e le vele staccabili di LoRA.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## Il giudice in formula

*Nel racconto: le bandierine; «scegliere è più facile che dare un voto».*

In [2]:
r_vincente, r_perdente = 2, 0
p = sigmoide(r_vincente - r_perdente)
controlla("probabilità che la persona scelga la vincente (%)", p * 100, 88, 0.5)
controlla("lo stesso 88% del mercato: softmax su (2; 0)", softmax([2, 0])[0], p, 1e-12)
print(f"   punti persi dal giudice su questa coppia: {-math.log(p):.2f}")

✓  probabilità che la persona scelga la vincente (%): calcolato 88.08, nel libro 88
✓  lo stesso 88% del mercato: softmax su (2; 0): calcolato 0.8808, nel libro 0.880797
   punti persi dal giudice su questa coppia: 0.13


## La cima

*Nel racconto: «legato da una cima a com'era prima».*

La KL divergence misura quanto le probabilità del nuovo modello si sono allontanate da quelle di prima. I numeri qui sotto sono un esempio, non stanno nel libro.

In [3]:
kl = lambda p, q: float(np.sum(np.array(p) * np.log(np.array(p) / np.array(q))))
prima = [0.66, 0.24, 0.10]
controlla("il modello è rimasto com'era: la cima è lenta", kl(prima, prima), 0)
print(f"   si è spostato un poco:  KL = {kl([0.70, 0.21, 0.09], prima):.3f}")
print(f"   si è spostato molto:    KL = {kl([0.98, 0.01, 0.01], prima):.3f}")

✓  il modello è rimasto com'era: la cima è lenta: calcolato 0, nel libro 0
   si è spostato un poco:  KL = 0.004
   si è spostato molto:    KL = 0.333


## La capriola e il fratello maggiore

*Nel racconto: il robot della capriola; il modello cento volte più piccolo.*

In [4]:
controlla("175 miliardi contro 1,3: quante volte più piccolo", 175 / 1.3, 135, 0.5)

✓  175 miliardi contro 1,3: quante volte più piccolo: calcolato 134.6, nel libro 135


## La soglia dell'esame

*Nel racconto: «giusta un punto, non lo so zero, sbagliata meno tre».*

In [5]:
guadagno = lambda p, k: p * 1 - (1 - p) * k              # rispondere, se l'errore costa k punti
soglia = lambda k: k / (1 + k)
controlla("con −3: conviene rispondere sopra", soglia(3), 0.75)
controlla("   al 75% rispondere e astenersi valgono uguale", guadagno(0.75, 3), 0, 1e-12)
t = 0.75
controlla("la formula letta al contrario: punti tolti con soglia t = 0,75", t / (1 - t), 3)

✓  con −3: conviene rispondere sopra: calcolato 0.75, nel libro 0.75
✓     al 75% rispondere e astenersi valgono uguale: calcolato 0, nel libro 0
✓  la formula letta al contrario: punti tolti con soglia t = 0,75: calcolato 3, nel libro 3


## L'arbitro e la media dei fratelli

*Nel racconto: «premiati solo sul risultato»; «ogni tentativo viene misurato contro la media dei suoi fratelli»; il sensore nel corpetto.*

In [6]:
premi = np.array([1, 1, 1, 0, 0, 0, 0, 0])              # 8 tentativi, 3 arrivano al risultato
media, deviazione = premi.mean(), premi.std()
controlla("media", media, 0.375, 0.0005)
controlla("deviazione standard", deviazione, 0.48)
controlla("   cioè √(0,375 · 0,625)", math.sqrt(0.375 * 0.625), 0.48)
controlla("ogni tentativo riuscito riceve", (1 - media) / deviazione, 1.3, 0.05)
controlla("ogni tentativo fallito riceve", (0 - media) / deviazione, -0.77)

✓  media: calcolato 0.375, nel libro 0.375
✓  deviazione standard: calcolato 0.4841, nel libro 0.48
✓     cioè √(0,375 · 0,625): calcolato 0.4841, nel libro 0.48
✓  ogni tentativo riuscito riceve: calcolato 1.291, nel libro 1.3
✓  ogni tentativo fallito riceve: calcolato -0.7746, nel libro -0.77


## Le vele staccabili

*Nel racconto: le vele nuove sullo stesso scafo; lo scafo compresso e fermo.*

In [7]:
d, rango = 4096, 8
conta("numeri di W", d * d, 16_777_216)
conta("numeri di B e A insieme", 2 * d * rango, 65_536)
controlla("frazione (%): «lo 0,4%» del racconto", 2 * d * rango / d ** 2 * 100, 0.39)

# il prodotto B A ha la misura di W, ma si muove solo lungo 8 direzioni
rng = np.random.default_rng(0)
B_, A_ = rng.standard_normal((256, rango)), rng.standard_normal((rango, 256))     # in piccolo: 256 al posto di 4.096
correzione = B_ @ A_
assert correzione.shape == (256, 256)
conta("direzioni lungo cui si muove la correzione", np.linalg.matrix_rank(correzione), rango)

controlla("70 miliardi di parametri a 16 bit (GB)", 70e9 * 2 / 1e9, 140)
controlla("a 4 bit (GB)", 70e9 * 0.5 / 1e9, 35)

✓  numeri di W: calcolato 16.777.216, nel libro 16.777.216
✓  numeri di B e A insieme: calcolato 65.536, nel libro 65.536
✓  frazione (%): «lo 0,4%» del racconto: calcolato 0.3906, nel libro 0.39
✓  direzioni lungo cui si muove la correzione: calcolato 8, nel libro 8
✓  70 miliardi di parametri a 16 bit (GB): calcolato 140, nel libro 140
✓  a 4 bit (GB): calcolato 35, nel libro 35


## Kata per casa

**Esercizio 2.** Giusta +1, «non lo so» 0, sbagliata −2. E se l'errore valesse −3?

In [8]:
controlla("soglia con errore −2 (%)", soglia(2) * 100, 67, 0.5)
controlla("soglia con errore −3 (%)", soglia(3) * 100, 75)
for sicurezza in [0.5, 0.7, 0.8]:
    print(f"sicuro al {sicurezza:.0%}, errore −3: rispondere vale {guadagno(sicurezza, 3):+.2f}, astenersi vale 0")
# un esame dove sbagliare non costa nulla rende conveniente rispondere sempre
controlla("soglia con k = 0", soglia(0), 0)

✓  soglia con errore −2 (%): calcolato 66.67, nel libro 67
✓  soglia con errore −3 (%): calcolato 75, nel libro 75
sicuro al 50%, errore −3: rispondere vale -1.00, astenersi vale 0
sicuro al 70%, errore −3: rispondere vale -0.20, astenersi vale 0
sicuro al 80%, errore −3: rispondere vale +0.20, astenersi vale 0
✓  soglia con k = 0: calcolato 0, nel libro 0
